# Two witnesses: $G_{13}$ and $G_{14}^a$

A graph is strong I-contractible if it reduces to one vertex by deleting vertices or edges with strong neighbourhoods. Homology means homology of the clique complex. We compute the first homology of $G_{13}$, compare $G_{14}^a$ with a one-edge extension, and read the two acyclicity columns of the table.

From the repository root, run `python3 tools/copy_notebook.py` and open the copy under `results/notebook/`. The cells use the Python standard library. Outputs are saved with that working copy.

In [ ]:
from pathlib import Path
import csv, json, subprocess, sys
sys.dont_write_bytecode = True
start = Path.cwd().resolve()
root = next((p for p in (start, *start.parents) if (p / "generate.py").is_file() and (p / "finite/certificates").is_dir()), None)
if root is None:
    raise RuntimeError("Open the notebook copy under this repository's results directory")
sys.path.insert(0, str(root / "finite/certificates"))
from finite_checks import decode, strong, h1
def export(name, extension=False):
    command = [sys.executable, str(root / "generate.py"), "--graph", name, "--format", "json"]
    if extension:
        command.append("--extension")
    return json.loads(subprocess.check_output(command, text=True))


## A 13-vertex graph with odd torsion

The clique complex of $G_{13}$ is obtained from a disc by attaching its boundary to a cycle with degree three. It has $H_1\cong\mathbb Z/3$ and vanishing reduced homology over $\mathbb F_2$. The following cell recomputes integral first homology from the triangle relations.

In [ ]:
g13 = export("G13")
assert (g13["n"], g13["m"]) == (13, 40)
assert h1(decode(g13["graph6"])) == (0, (3,))
print("G13:", g13["n"], "vertices,", g13["m"], "edges; H1 = Z/3")


## One edge changes strong contractibility

The 44-edge graph $G_{14}^a$ is not strong I-contractible, but adding $\{0,1\}$ gives a strong graph. The cell below checks these two membership statements. The finite reproduction command also verifies that the common neighbourhood of 0 and 1 is strong, making the reverse deletion admissible. This 14-vertex extension refutes the minimum of 15 proposed in Conjecture 15 of Dochtermann et al. (2023).

In [ ]:
ga = export("G14a")
plus = export("G14a", extension=True)
assert (ga["n"], ga["m"], plus["m"]) == (14, 44, 45)
assert not strong(decode(ga["graph6"]))
assert strong(decode(plus["graph6"]))
print("G14a is non-strong; its one-edge extension is strong.")


## The two coefficient columns

Rows 3–13 count all connected graphs up to isomorphism; every column of row 14 counts only connected $K_4$-free graphs. In these universes, the $\mathbb F_2$-acyclic graphs outside the strong class split into graphs certified I-contractible and graphs with first integral homology $\mathbb Z/3$. Thus integral acyclicity and I-contractibility coincide there. The difference between the two acyclicity columns is the number of torsion graphs: three at order 13 and 625 in row 14.

The next cell reads those differences from the supplied table.

In [ ]:
with (root / "data/cardinalities.csv").open(newline="") as stream:
    rows = list(csv.DictReader(stream))
for row in rows[-2:]:
    gap = int(row["acyclic_F2"]) - int(row["acyclic_Z"])
    assert gap == (3 if row["n"] == "13" else 625)
    print("n =", row["n"], "universe =", row["universe"], "F2/Z gap =", gap)


## Verification and proofs

Run `python3 reproduce.py` from the repository root to check the finite witnesses, certificates and numerical sources. Add `--extended` to compare the C classifier with direct mathematical recurrences on every labelled graph of order at most five; this requires a C11 compiler.

[Article](https://github.com/Geometric-and-Combinatorial-Structures/minimal-I-contractible-not-strong/blob/main/paper/main.pdf) · [Mathematical supplement](https://github.com/Geometric-and-Combinatorial-Structures/minimal-I-contractible-not-strong/blob/main/supplement/main.pdf) · [Full enumeration](https://github.com/Geometric-and-Combinatorial-Structures/minimal-I-contractible-not-strong/blob/main/docs/full-reproduction.md)